# Homework 5. CTE, Recursive CTE & Materialized Views
## Task 1. Setup and Typed Schema

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

import os
import pandas as pd
import pgserver
import kagglehub
from sqlalchemy import create_engine, text

# Start PostgreSQL 16
pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

# Check PostgreSQL version
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 65.9 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
# Download the Olist dataset from Kaggle
dataset_dir = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce"
)

print("Dataset directory:", dataset_dir)

# Display available CSV files
for filename in sorted(os.listdir(dataset_dir)):
    if filename.endswith(".csv"):
        print(filename)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset directory: /kaggle/input/brazilian-ecommerce
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
# Map raw tables to source CSV files
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
}

# Load CSV files into PostgreSQL
for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)

    df = pd.read_csv(path)

    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False,
        chunksize=10_000
    )

    print(f"{table_name}: {df.shape[0]} rows")

olist_customers_raw: 99441 rows
olist_orders_raw: 99441 rows
olist_order_items_raw: 112650 rows
olist_products_raw: 32951 rows
olist_sellers_raw: 3095 rows


In [4]:
# Create typed Olist tables
typed_ddl = """
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

DROP TABLE IF EXISTS hw5_referrals CASCADE;

DROP TABLE IF EXISTS
    olist_order_items,
    olist_orders,
    olist_customers,
    olist_products,
    olist_sellers
CASCADE;

CREATE TABLE olist_customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state CHAR(2) NOT NULL
);

CREATE TABLE olist_products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length INTEGER
        CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER
        CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty INTEGER
        CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g INTEGER
        CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm INTEGER
        CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm INTEGER
        CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm INTEGER
        CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city TEXT,
    seller_state CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL
        REFERENCES olist_customers(customer_id),
    order_status TEXT NOT NULL,
    order_purchase_timestamp TIMESTAMP NOT NULL,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,

    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_order_items (
    order_id TEXT NOT NULL
        REFERENCES olist_orders(order_id),
    order_item_id INTEGER NOT NULL,
    product_id TEXT
        REFERENCES olist_products(product_id),
    seller_id TEXT
        REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price NUMERIC(12, 2)
        CHECK (price IS NULL OR price >= 0),
    freight_value NUMERIC(12, 2)
        CHECK (freight_value IS NULL OR freight_value >= 0),

    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX hw5_idx_orders_customer
    ON olist_orders(customer_id);

CREATE INDEX hw5_idx_items_product
    ON olist_order_items(product_id);
"""

with engine.begin() as conn:
    conn.execute(text(typed_ddl))

print("Typed tables created successfully.")

Typed tables created successfully.


In [5]:
# Insert data into typed tables
insert_sql = """
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;
"""

with engine.begin() as conn:
    conn.execute(text(insert_sql))

print("Typed data inserted successfully.")

Typed data inserted successfully.


In [6]:
# Compare row counts between raw and typed tables
df = pd.read_sql(text("""
    SELECT
        'customers' AS table_name,
        (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows,
        (SELECT COUNT(*) FROM olist_customers) AS typed_rows

    UNION ALL

    SELECT
        'orders',
        (SELECT COUNT(*) FROM olist_orders_raw),
        (SELECT COUNT(*) FROM olist_orders)

    UNION ALL

    SELECT
        'order_items',
        (SELECT COUNT(*) FROM olist_order_items_raw),
        (SELECT COUNT(*) FROM olist_order_items)

    UNION ALL

    SELECT
        'products',
        (SELECT COUNT(*) FROM olist_products_raw),
        (SELECT COUNT(*) FROM olist_products)

    UNION ALL

    SELECT
        'sellers',
        (SELECT COUNT(*) FROM olist_sellers_raw),
        (SELECT COUNT(*) FROM olist_sellers)

    ORDER BY table_name
"""), engine)

display(df)

assert (df["raw_rows"] == df["typed_rows"]).all()

print("All row counts match.")

,table_name,raw_rows,typed_rows
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,sellers,3095,3095


All row counts match.


### Результат

Було завантажено п'ять таблиць із датасету Olist: покупці, замовлення, позиції замовлень, товари та продавці.

Спочатку дані завантажувалися в raw-таблиці, а потім переносилися в окремі типізовані таблиці через `INSERT INTO ... SELECT`.

У typed-таблицях було додано первинні й зовнішні ключі, а також обмеження для перевірки значень. Наприклад, ціна товару не може бути від'ємною.

Після завантаження було порівняно кількість записів у raw- та typed-таблицях. Це дозволило перевірити, чи всі дані були перенесені.

Підготовлені таблиці будуть використовуватися для розрахунку ознак покупців через CTE.

## Task 2. CTE-chain Feature Engineering Pipeline

У цій частині роботи було створено SQL-pipeline для розрахунку ознак покупців.

Для цього використано CTE, які дозволяють розділити складний запит на кілька простіших етапів. Кожен етап виконує окреме завдання, наприклад підрахунок суми замовлень або визначення улюбленої категорії товарів.

У результаті для кожного `customer_id` будуть розраховані RFM-ознаки, квартал першого замовлення, середня вартість замовлень за 30 днів, частка скасованих замовлень та найпопулярніша категорія товарів.

Для розрахунку давності останнього замовлення використовується дата з датасету, а не поточна дата. Це потрібно, щоб результати не змінювалися з часом.

In [7]:
# Define the complete customer feature engineering pipeline
feature_pipeline_sql = """
WITH
reference_date AS (
    -- Stage 0: Use a stable reference date from the dataset
    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),

stage1_orders_total AS (
    -- Stage 1: Calculate the total amount of each order
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(
            SUM(
                COALESCE(oi.price, 0)
                + COALESCE(oi.freight_value, 0)
            ), 0
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON o.order_id = oi.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),

stage2_rfm AS (
    -- Stage 2: Calculate recency, frequency, and monetary value
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER
            AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY
        s.customer_id,
        rd.as_of_date
),

stage3_cohort AS (
    -- Stage 3: Find the quarter of the first purchase
    SELECT
        r.*,
        DATE_TRUNC(
            'quarter',
            r.first_order_date
        )::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),

stage4_rolling_aov AS (
    -- Stage 4: Calculate rolling 30-day average order value
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,

        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN
                INTERVAL '30 days' PRECEDING
                AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,

        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent

    FROM stage1_orders_total AS s
),

stage5_cancel_rate AS (
    -- Stage 5: Calculate the cancellation rate
    SELECT
        customer_id,
        AVG(
            CASE
                WHEN order_status = 'canceled' THEN 1.0
                ELSE 0.0
            END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),

stage6_category_revenue AS (
    -- Stage 6: Calculate revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(
            p.product_category_name,
            '(unknown)'
        ) AS product_category_name,

        SUM(
            COALESCE(oi.price, 0)
            + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue

    FROM olist_orders AS o
    INNER JOIN olist_order_items AS oi
        ON o.order_id = oi.order_id
    LEFT JOIN olist_products AS p
        ON oi.product_id = p.product_id
    GROUP BY
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)')
),

stage7_preferred_category AS (
    -- Stage 7: Select the highest-revenue category per customer
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY
                    cr.category_revenue DESC,
                    cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)

-- Combine all features into one row per customer
SELECT
    c.customer_id,
    c.customer_state,

    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,

    coh.first_order_date,
    coh.cohort_quarter,

    COALESCE(
        aov.avg_order_value_30d, 0
    )::NUMERIC(12, 2) AS avg_order_value_30d_recent,

    COALESCE(
        cancel.cancellation_rate, 0
    )::NUMERIC(6, 4) AS cancellation_rate,

    COALESCE(
        pc.product_category_name,
        '(none)'
    ) AS preferred_category,

    COALESCE(
        pc.category_revenue, 0
    )::NUMERIC(12, 2) AS preferred_category_revenue

FROM olist_customers AS c

LEFT JOIN stage3_cohort AS coh
    ON c.customer_id = coh.customer_id

LEFT JOIN stage4_rolling_aov AS aov
    ON c.customer_id = aov.customer_id
    AND aov.rn_recent = 1

LEFT JOIN stage5_cancel_rate AS cancel
    ON c.customer_id = cancel.customer_id

LEFT JOIN stage7_preferred_category AS pc
    ON c.customer_id = pc.customer_id
"""

print("CTE pipeline defined successfully.")

CTE pipeline defined successfully.


In [8]:
# Execute the pipeline and preview customers with highest spending
features_df = pd.read_sql(
    text(feature_pipeline_sql + """
        ORDER BY monetary DESC NULLS LAST, customer_id
        LIMIT 20
    """),
    engine
)

display(features_df)

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0,pcs,4681.78


In [9]:
# Validate the final feature dataset
validation_df = pd.read_sql(
    text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT customer_id) AS unique_customers,
            COUNT(*) - COUNT(DISTINCT customer_id) AS duplicates,

            COUNT(*) FILTER (
                WHERE recency_days < 0
            ) AS invalid_recency,

            COUNT(*) FILTER (
                WHERE cancellation_rate < 0
                   OR cancellation_rate > 1
            ) AS invalid_cancellation_rate

        FROM (
    """ + feature_pipeline_sql + """
        ) AS features
    """),
    engine
)

display(validation_df)

assert validation_df.iloc[0]["duplicates"] == 0
assert validation_df.iloc[0]["invalid_recency"] == 0
assert validation_df.iloc[0]["invalid_cancellation_rate"] == 0
assert validation_df.iloc[0]["total_rows"] == validation_df.iloc[0]["unique_customers"]

print("Feature pipeline validation passed.")

,total_rows,unique_customers,duplicates,invalid_recency,invalid_cancellation_rate
0,99441,99441,0,0,0


Feature pipeline validation passed.


In [10]:
# Summarize customer features
summary_df = pd.read_sql(
    text("""
        SELECT
            COUNT(*) AS customers,
            ROUND(AVG(recency_days), 2) AS avg_recency_days,
            ROUND(AVG(frequency), 2) AS avg_frequency,
            ROUND(AVG(monetary), 2) AS avg_monetary,
            ROUND(AVG(cancellation_rate), 4) AS avg_cancellation_rate
        FROM (
    """ + feature_pipeline_sql + """
        ) AS features
    """),
    engine
)

display(summary_df)

,customers,avg_recency_days,avg_frequency,avg_monetary,avg_cancellation_rate
0,99441,291.27,1.0,159.33,0.0063


### Результат

Для розрахунку ознак було створено CTE-chain із семи основних етапів.

Спочатку було розраховано загальну вартість кожного замовлення. На основі цих даних отримано RFM-ознаки: давність останнього замовлення, кількість замовлень і загальну суму витрат.

Далі було визначено квартал першої покупки та розраховано середню вартість замовлень за 30 днів за допомогою window function.

Окремо було підраховано частку скасованих замовлень і визначено категорію товарів, на яку покупець витратив найбільше коштів.

Для вибору категорії використано `ROW_NUMBER()`, а для розрахунку середньої вартості замовлень - `AVG() OVER`.

Після об'єднання результатів було перевірено, що кожен `customer_id` має тільки один рядок і значення основних ознак не виходять за допустимі межі.

CTE дозволили розділити складні обчислення на окремі етапи, тому запит стало простіше читати та перевіряти.

### Аналіз отриманих результатів

У результаті виконання CTE-pipeline було отримано ознаки для 99 441 запису покупців.

Середня давність останнього замовлення становить 291.27 дня, а середня сума витрат - 159.33 BRL.

Середня кількість замовлень дорівнює 1.00. Це пояснюється особливістю датасету Olist: кожне замовлення має окремий `customer_id`. Для аналізу повторних покупок краще використовувати `customer_unique_id`, як у попередньому домашньому завданні.

Середня частка скасованих замовлень становить 0.0063, або приблизно 0.63%.

Отримані результати показують, що SQL-pipeline працює, але деякі ознаки, особливо `frequency`, будуть більш корисними для ML, якщо розраховувати їх на рівні реального покупця, а не окремого замовлення.

## Task 3. Recursive CTE - Referral Chain

In [11]:
# Create a synthetic referral table
referrals_ddl = """
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL
        REFERENCES olist_customers(customer_id),

    referred_customer_id TEXT NOT NULL
        REFERENCES olist_customers(customer_id),

    referred_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    PRIMARY KEY (referred_customer_id),

    CHECK (referrer_customer_id <> referred_customer_id)
);
"""

with engine.begin() as conn:
    conn.execute(text(referrals_ddl))

print("Referral table created successfully.")

Referral table created successfully.


In [12]:
# Generate a referral chain using the first 10 customers
insert_referrals_sql = """
WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM (
        SELECT customer_id
        FROM olist_customers
        ORDER BY customer_id
        LIMIT 10
    ) AS selected
)

INSERT INTO hw5_referrals (
    referrer_customer_id,
    referred_customer_id
)
SELECT
    prev.customer_id,
    nxt.customer_id
FROM ordered_customers AS prev
INNER JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1

ON CONFLICT (referred_customer_id) DO NOTHING;
"""

with engine.begin() as conn:
    conn.execute(text(insert_referrals_sql))

print("Synthetic referrals inserted successfully.")

Synthetic referrals inserted successfully.


In [13]:
# View the synthetic referral relationships
df = pd.read_sql(text("""
    SELECT
        referrer_customer_id,
        referred_customer_id,
        referred_at
    FROM hw5_referrals
    ORDER BY referrer_customer_id
"""), engine)

display(df)

print(f"Total referral relationships: {len(df)}")

,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-15 19:09:03.391159+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-15 19:09:03.391159+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-15 19:09:03.391159+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-15 19:09:03.391159+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-15 19:09:03.391159+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-15 19:09:03.391159+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-15 19:09:03.391159+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-15 19:09:03.391159+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-15 19:09:03.391159+00:00


Total referral relationships: 9


In [14]:
# Calculate referral depth and path using a recursive CTE
recursive_sql = """
WITH RECURSIVE referral_depth AS (

    -- Anchor: Find customers without a referrer
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text

    FROM olist_customers AS c

    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- Recursive: Follow referral relationships
    SELECT
        r.referred_customer_id,
        rd.depth + 1,
        rd.path_ids || r.referred_customer_id,
        rd.path_text || ' -> ' || r.referred_customer_id

    FROM hw5_referrals AS r

    INNER JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id

    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)

SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
"""

referral_df = pd.read_sql(text(recursive_sql), engine)

display(referral_df)

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


In [15]:
# Display the synthetic referral chain from its root
chain_df = pd.read_sql(text("""
    WITH RECURSIVE referral_chain AS (

        -- Anchor: Start from the root of the synthetic chain
        SELECT
            c.customer_id,
            1 AS depth,
            ARRAY[c.customer_id]::TEXT[] AS path_ids,
            c.customer_id::TEXT AS path_text

        FROM olist_customers AS c

        WHERE c.customer_id = (
            SELECT MIN(referrer_customer_id)
            FROM hw5_referrals
        )

        UNION ALL

        -- Recursive: Follow each referral
        SELECT
            r.referred_customer_id,
            rc.depth + 1,
            rc.path_ids || r.referred_customer_id,
            rc.path_text || ' -> ' || r.referred_customer_id

        FROM hw5_referrals AS r

        INNER JOIN referral_chain AS rc
            ON r.referrer_customer_id = rc.customer_id

        WHERE NOT r.referred_customer_id = ANY(rc.path_ids)
          AND rc.depth < 50
    )

    SELECT
        customer_id,
        depth,
        path_text
    FROM referral_chain
    ORDER BY depth
"""), engine)

display(chain_df)

,customer_id,depth,path_text
0,00012a2ce6f8dcda20d059ce98491703,1,00012a2ce6f8dcda20d059ce98491703
1,000161a058600d5901f007fab4c27140,2,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,0001fd6190edaaf884bcaf3d49edf079,3,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,0002414f95344307404f0ace7a26f1d5,4,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,000379cdec625522490c315e70c7a9fb,5,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,0004164d20a9e969af783496f3408652,6,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,000419c5494106c306a97b5635748086,7,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,00046a560d407e99b969756e0b10f282,8,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,00050bf6e01e69d5c0fd612f1bcfb69c,9,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,000598caf2ef4117407665ac33275130,10,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


In [16]:
# Verify referral chain depth and record count
assert len(chain_df) == 10
assert chain_df["depth"].min() == 1
assert chain_df["depth"].max() == 10
assert chain_df["customer_id"].is_unique

print("Recursive CTE verification passed.")

Recursive CTE verification passed.


### Результат

Для перевірки recursive CTE було створено 9 зв'язків між 10 покупцями. Отримано ланцюжок із рівнями від 1 до 10.

Запит складається з двох частин. Anchor знаходить початкових покупців, яких ніхто не запросив. Recursive part переходить від одного покупця до наступного та збільшує `depth`.

Для кожного запису також формується `path`, який показує весь шлях від першого покупця.

Щоб запит не виконувався нескінченно, було додано два обмеження: перевірку вже відвіданих покупців через `path_ids` та максимальну глибину 50.

Наприклад, якщо виникне цикл `A → B → A`, перевірка `path_ids` не дозволить повторно перейти до `A`.

У результаті recursive CTE дозволяє працювати з ієрархічними даними та безпечно обходити referral-мережу.

## Task 4. Materialized View with Concurrent Refresh

In [17]:
# Create Materialized View using the full CTE pipeline
create_mv_sql = """
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
""" + feature_pipeline_sql + """;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);
"""

with engine.begin() as conn:
    conn.execute(text(create_mv_sql))

print("Materialized View created successfully.")

Materialized View created successfully.


In [18]:
# Verify Materialized View row count
mv_count_df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT customer_id) AS unique_customers
    FROM mv_olist_customer_features
"""), engine)

display(mv_count_df)

assert mv_count_df.iloc[0]["total_rows"] == 99441
assert mv_count_df.iloc[0]["unique_customers"] == 99441

print("Materialized View validation passed.")

,total_rows,unique_customers
0,99441,99441


Materialized View validation passed.


In [19]:
# Preview Materialized View features
mv_df = pd.read_sql(text("""
    SELECT
        customer_id,
        customer_state,
        recency_days,
        frequency,
        monetary,
        cohort_quarter,
        avg_order_value_30d_recent,
        cancellation_rate,
        preferred_category
    FROM mv_olist_customer_features
    ORDER BY monetary DESC NULLS LAST, customer_id
    LIMIT 5
"""), engine)

display(mv_df)

,customer_id,customer_state,recency_days,frequency,monetary,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-07-01,13664.08,0.0,telefonia_fixa
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-01,7274.88,0.0,telefonia_fixa
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-01-01,6929.31,0.0,utilidades_domesticas
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-01,6922.21,0.0,pcs
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-04-01,6726.66,0.0,artes


In [20]:
# Check indexes on the Materialized View
index_df = pd.read_sql(text("""
    SELECT
        indexname,
        indexdef
    FROM pg_indexes
    WHERE tablename = 'mv_olist_customer_features'
"""), engine)

display(index_df)

,indexname,indexdef
0,idx_mv_olist_customer_features_customer_id,CREATE UNIQUE INDEX idx_mv_olist_customer_feat...


In [21]:
# Refresh the Materialized View concurrently
refresh_sql = """
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;
"""

with engine.connect().execution_options(
    isolation_level="AUTOCOMMIT"
) as conn:
    conn.execute(text(refresh_sql))

print("Materialized View refreshed successfully.")

Materialized View refreshed successfully.


In [22]:
# Verify Materialized View after refresh
df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_customers,
        ROUND(AVG(recency_days), 2) AS avg_recency,
        ROUND(AVG(frequency), 2) AS avg_frequency,
        ROUND(AVG(monetary), 2) AS avg_monetary,
        ROUND(AVG(cancellation_rate), 4) AS avg_cancellation_rate
    FROM mv_olist_customer_features
"""), engine)

display(df)

,total_customers,avg_recency,avg_frequency,avg_monetary,avg_cancellation_rate
0,99441,291.27,1.0,159.33,0.0063


### Результат

На основі CTE-pipeline було створено Materialized View `mv_olist_customer_features`.

Він містить усі розраховані ознаки для 99 441 запису покупців. Для швидкого пошуку за `customer_id` було додано унікальний індекс.

Після створення було виконано `REFRESH MATERIALIZED VIEW CONCURRENTLY`. Ця команда дозволяє оновлювати дані, не блокуючи звичайне читання Materialized View.

Після оновлення було повторно перевірено кількість записів і середні значення ознак. Результати залишилися такими самими, як у CTE-pipeline.

Materialized View дозволяє зберігати готові результати обчислень і швидше отримувати їх під час повторних запитів. Але для підтримки актуальності даних потрібно періодично виконувати оновлення.

## Task 5. EXPLAIN ANALYZE + Reflection

In [23]:
# Analyze the execution plan of the complete CTE pipeline
explain_cte_sql = """
EXPLAIN (ANALYZE, BUFFERS)
""" + feature_pipeline_sql + """
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 10
"""

cte_plan_df = pd.read_sql(text(explain_cte_sql), engine)

# Display the full execution plan
for line in cte_plan_df["QUERY PLAN"]:
    print(line)

Limit  (cost=115110.81..115110.83 rows=10 width=150) (actual time=4259.878..4262.817 rows=10 loops=1)
  Buffers: shared hit=111711, temp read=8440 written=8589
  CTE stage1_orders_total
    ->  GroupAggregate  (cost=0.83..27070.13 rows=99441 width=103) (actual time=0.061..362.518 rows=99441 loops=1)
          Group Key: o_1.order_id
          Buffers: shared hit=103453
          ->  Merge Left Join  (cost=0.83..24485.04 rows=112650 width=95) (actual time=0.040..228.248 rows=113425 loops=1)
                Merge Cond: (o_1.order_id = oi_1.order_id)
                Buffers: shared hit=103453
                ->  Index Scan using olist_orders_pkey on olist_orders o_1  (cost=0.42..12564.02 rows=99441 width=83) (actual time=0.019..93.355 rows=99441 loops=1)
                      Buffers: shared hit=100347
                ->  Index Scan using olist_order_items_pkey on olist_order_items oi_1  (cost=0.42..10264.29 rows=112650 width=45) (actual time=0.013..34.245 rows=112650 loops=1)
           

In [24]:
# Analyze customer lookup performance in the Materialized View
explain_mv_sql = """
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id
"""

mv_plan_df = pd.read_sql(text(explain_mv_sql), engine)

# Display the full execution plan
for line in mv_plan_df["QUERY PLAN"]:
    print(line)

Nested Loop  (cost=0.83..8.91 rows=1 width=92) (actual time=0.058..0.060 rows=1 loops=1)
  Buffers: shared hit=8
  ->  Limit  (cost=0.42..0.47 rows=1 width=33) (actual time=0.025..0.026 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..5328.03 rows=99441 width=33) (actual time=0.024..0.024 rows=1 loops=1)
              Heap Fetches: 0
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=92) (actual time=0.026..0.026 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=4
Planning:
  Buffers: shared hit=7 read=1
Planning Time: 0.621 ms
Execution Time: 0.102 ms


In [25]:
import re

# Extract execution time from an EXPLAIN ANALYZE result
def get_execution_time(plan_df):
    for line in plan_df["QUERY PLAN"]:
        match = re.search(r"Execution Time:\s*([\d.]+)\s*ms", line)
        if match:
            return float(match.group(1))
    return None

cte_time = get_execution_time(cte_plan_df)
mv_time = get_execution_time(mv_plan_df)

comparison_df = pd.DataFrame({
    "Query": ["Full CTE pipeline (top 10)", "MV customer lookup"],
    "Execution Time (ms)": [cte_time, mv_time]
})

display(comparison_df)

if cte_time is not None and mv_time is not None and mv_time > 0:
    print(f"Execution time ratio: {cte_time / mv_time:.1f}x")

,Query,Execution Time (ms)
0,Full CTE pipeline (top 10),4271.135
1,MV customer lookup,0.102


Execution time ratio: 41873.9x


In [26]:
# Compare the full pipeline for a single customer lookup
explain_cte_lookup_sql = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM (
""" + feature_pipeline_sql + """
) AS features
WHERE customer_id = (
    SELECT MIN(customer_id)
    FROM olist_customers
)
"""

cte_lookup_plan_df = pd.read_sql(
    text(explain_cte_lookup_sql),
    engine
)

for line in cte_lookup_plan_df["QUERY PLAN"]:
    print(line)

Nested Loop Left Join  (cost=34696.73..36969.68 rows=2 width=150) (actual time=715.330..715.362 rows=1 loops=1)
  Buffers: shared hit=105277 read=4, temp read=2646 written=1323
  InitPlan 2 (returns $1)
    ->  Result  (cost=0.47..0.48 rows=1 width=32) (actual time=0.032..0.034 rows=1 loops=1)
          Buffers: shared hit=4
          InitPlan 1 (returns $0)
            ->  Limit  (cost=0.42..0.47 rows=1 width=33) (actual time=0.028..0.030 rows=1 loops=1)
                  Buffers: shared hit=4
                  ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..5576.63 rows=99441 width=33) (actual time=0.027..0.028 rows=1 loops=1)
                        Index Cond: (customer_id IS NOT NULL)
                        Heap Fetches: 0
                        Buffers: shared hit=4
  CTE stage1_orders_total
    ->  GroupAggregate  (cost=0.83..27070.13 rows=99441 width=103) (actual time=0.098..520.856 rows=99441 loops=1)
          Group Key: o_1.order_id
         

In [27]:
# Compare two ways of retrieving the same customer features
cte_lookup_time = get_execution_time(cte_lookup_plan_df)

lookup_comparison_df = pd.DataFrame({
    "Method": ["CTE lookup", "Materialized View lookup"],
    "Execution Time (ms)": [cte_lookup_time, mv_time]
})

display(lookup_comparison_df)

if cte_lookup_time is not None and mv_time is not None and mv_time > 0:
    print(
        f"MV lookup speedup for this execution: "
        f"{cte_lookup_time / mv_time:.1f}x"
    )

,Method,Execution Time (ms)
0,CTE lookup,717.488
1,Materialized View lookup,0.102


MV lookup speedup for this execution: 7034.2x


In [28]:
# Display important execution plan operations
keywords = [
    "HashAggregate",
    "GroupAggregate",
    "Aggregate",
    "Sort",
    "Hash Join",
    "Merge Join",
    "Nested Loop",
    "WindowAgg",
    "Seq Scan",
    "Index Scan",
    "Buffers:",
    "Execution Time:"
]

for line in cte_plan_df["QUERY PLAN"]:
    if any(keyword in line for keyword in keywords):
        print(line)

  Buffers: shared hit=111711, temp read=8440 written=8589
    ->  GroupAggregate  (cost=0.83..27070.13 rows=99441 width=103) (actual time=0.061..362.518 rows=99441 loops=1)
          Buffers: shared hit=103453
                Buffers: shared hit=103453
                ->  Index Scan using olist_orders_pkey on olist_orders o_1  (cost=0.42..12564.02 rows=99441 width=83) (actual time=0.019..93.355 rows=99441 loops=1)
                      Buffers: shared hit=100347
                ->  Index Scan using olist_order_items_pkey on olist_order_items oi_1  (cost=0.42..10264.29 rows=112650 width=45) (actual time=0.013..34.245 rows=112650 loops=1)
                      Buffers: shared hit=3106
  ->  Sort  (cost=88040.68..88289.28 rows=99441 width=150) (actual time=4259.876..4262.810 rows=10 loops=1)
        Sort Key: ((COALESCE(r.monetary, '0'::numeric))::numeric(12,2)) DESC NULLS LAST, c.customer_id
        Sort Method: top-N heapsort  Memory: 26kB
        Buffers: shared hit=111711, temp read=8

### Результати порівняння

Було виконано `EXPLAIN (ANALYZE, BUFFERS)` для повного CTE-pipeline та пошуку одного покупця через Materialized View.

Також було додатково перевірено пошук одного покупця через повний CTE-pipeline, щоб порівняти два способи отримання однакових ознак.

Для аналізу враховуються час виконання запитів, операції з'єднання та групування таблиць, сортування й використання індексів.

На основі отриманих планів можна оцінити, які обчислення потребують найбільше ресурсів і в яких випадках доцільно використовувати Materialized View.

### Аналіз результатів EXPLAIN

Повний CTE-pipeline виконувався 4271.135 мс, а пошук одного покупця через Materialized View - лише 0.102 мс.

Для більш точного порівняння було також виконано пошук одного покупця через CTE. Він зайняв 717.488 мс, тобто в цьому запуску Materialized View працював приблизно у 7034 рази швидше.

У плані виконання CTE було видно операції `GroupAggregate`, `Hash Join`, `WindowAgg` та `Sort`. Особливо складним виявився розрахунок найпопулярнішої категорії товарів, для якого потрібно об'єднувати, групувати та сортувати багато записів.

Також частина сортування використовувала тимчасові файли на диску.

Отже, Materialized View значно прискорює повторне читання готових ознак. Але для цього потрібно додаткове місце для зберігання результатів і періодичне оновлення даних.

## 6. Reflection

**1. Які обчислення були найдорожчими?**

За результатами `EXPLAIN ANALYZE`, повний CTE-pipeline виконувався 4271.135 мс. Одним із найскладніших етапів був розрахунок `preferred_category`, оскільки він потребує декількох `JOIN`, групування та сортування великої кількості записів. У плані були операції `GroupAggregate`, `WindowAgg`, `Hash Join` та `Sort`. Частина сортування використовувала тимчасові файли на диску, що також могло вплинути на швидкість.

**2. Чи варто використовувати Materialized View?**

Так, для цього pipeline Materialized View виглядає корисним рішенням. Пошук одного покупця через CTE зайняв 717.488 мс, а через MV - лише 0.102 мс. Це велика різниця, хоча результати отримано лише за один запуск. Для такого історичного датасету достатньо оновлення on-demand. У реальній системі з регулярними покупками можна було б оновлювати ознаки щодоби.

**3. Коли краще використовувати звичайний VIEW?**

Звичайний VIEW більше підходить, коли важливо завжди отримувати найновіші дані. Наприклад, для перегляду поточного статусу замовлення. VIEW не потребує окремого оновлення, але може виконуватися повільніше через повторні обчислення.

**4. Як працює захист від циклів у recursive CTE?**

У recursive CTE використано масив `path_ids`, який зберігає вже відвіданих покупців. Якщо виникне цикл `A → B → A`, повторний перехід до `A` не відбудеться. Додатково встановлено обмеження глибини 50 рівнів, щоб рекурсія не продовжувалася занадто довго.

**5. Який компроміс найважливіший?**

Для цього сценарію найважливішими є швидкість читання (latency) та вартість повторних обчислень (recompute-cost). Materialized View дозволяє зменшити обидві проблеми, але дані можуть бути не повністю актуальними. Тому частоту оновлення потрібно обирати залежно від того, наскільки свіжі ознаки потрібні ML-моделі.